# MNIST Handwritten Digit Classification using TensorFlow/Keras

**Assignment:** Build a simple neural network to classify handwritten digits (0–9).

This notebook covers:
1. Loading and exploring MNIST
2. Displaying sample images
3. Building, compiling, and training a neural network
4. Evaluating test accuracy
5. Visualizing training/validation accuracy and loss
6. Testing 5 handwritten images and comparing actual vs predicted labels
7. One experiment: adding Dropout and comparing the results

**Dataset:** MNIST contains 60,000 training images and 10,000 test images. Each image is a 28×28 grayscale image with a label from 0 to 9.


In [ ]:
# Install dependencies if needed
# !pip install tensorflow matplotlib numpy pandas seaborn

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print("TensorFlow version:", tf.__version__)


## 1. Load and Explore the MNIST Dataset

MNIST is provided directly through Keras. The training set contains 60,000 28×28 grayscale images and the test set contains 10,000 images.


In [ ]:
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

print("Training images shape:", x_train.shape)
print("Training labels shape:", y_train.shape)
print("Test images shape:", x_test.shape)
print("Test labels shape:", y_test.shape)
print("Pixel value range:", x_train.min(), "to", x_train.max())
print("Number of classes:", len(np.unique(y_train)))
print("Classes:", np.unique(y_train))


In [ ]:
plt.figure(figsize=(10, 5))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(x_train[i], cmap="gray")
    plt.title(f"Label: {y_train[i]}")
    plt.axis("off")
plt.tight_layout()
plt.show()


## 2. Preprocess the Data

Pixel values range from 0 to 255. Dividing by 255 scales them to 0–1, which is more suitable for neural-network training.


In [ ]:
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0
print("Normalized training range:", x_train.min(), "to", x_train.max())


## 3. Build, Compile, and Train the Baseline Neural Network

Architecture: Flatten → Dense(128, ReLU) → Dense(10, Softmax). The model uses Adam, sparse categorical cross-entropy, and 5 epochs.


In [ ]:
baseline_model = keras.Sequential([
    layers.Input(shape=(28, 28)),
    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dense(10, activation="softmax")
])

baseline_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
baseline_model.summary()


In [ ]:
baseline_history = baseline_model.fit(
    x_train, y_train,
    epochs=5,
    batch_size=128,
    validation_split=0.10,
    verbose=1
)


## 4. Evaluate the Baseline Model


In [ ]:
baseline_test_loss, baseline_test_accuracy = baseline_model.evaluate(x_test, y_test, verbose=0)
print(f"Baseline test loss: {baseline_test_loss:.4f}")
print(f"Baseline test accuracy: {baseline_test_accuracy:.4%}")


## 5. Visualize Training and Validation Accuracy/Loss


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(baseline_history.history["accuracy"], label="Training Accuracy")
plt.plot(baseline_history.history["val_accuracy"], label="Validation Accuracy")
plt.xlabel("Epoch"); plt.ylabel("Accuracy")
plt.title("Baseline Model: Training vs Validation Accuracy")
plt.legend(); plt.grid(True); plt.show()

plt.figure(figsize=(8, 5))
plt.plot(baseline_history.history["loss"], label="Training Loss")
plt.plot(baseline_history.history["val_loss"], label="Validation Loss")
plt.xlabel("Epoch"); plt.ylabel("Loss")
plt.title("Baseline Model: Training vs Validation Loss")
plt.legend(); plt.grid(True); plt.show()


## 6. Test the Model on 5 Handwritten Images

Five images are selected from the MNIST test set. They were not used during training. The table compares actual and predicted labels.


In [ ]:
sample_indices = [0, 1, 2, 3, 4]
sample_images = x_test[sample_indices]
sample_actual = y_test[sample_indices]
sample_probabilities = baseline_model.predict(sample_images, verbose=0)
sample_predictions = np.argmax(sample_probabilities, axis=1)

comparison = pd.DataFrame({
    "Image": [f"Image {i+1}" for i in range(5)],
    "Actual Label": sample_actual,
    "Predicted Label": sample_predictions,
    "Correct": sample_actual == sample_predictions
})
display(comparison)


In [ ]:
plt.figure(figsize=(12, 3))
for position, (image, actual, predicted) in enumerate(zip(sample_images, sample_actual, sample_predictions), start=1):
    plt.subplot(1, 5, position)
    plt.imshow(image, cmap="gray")
    plt.title(f"Actual: {actual}\nPredicted: {predicted}")
    plt.axis("off")
plt.tight_layout(); plt.show()

five_image_accuracy = np.mean(sample_actual == sample_predictions)
print(f"Accuracy on these 5 selected images: {five_image_accuracy:.0%}")


## 7. Experiment: Add Dropout

A Dropout layer with rate 0.20 is added after the hidden Dense layer. This is intended to reduce overfitting and improve generalization.


In [ ]:
dropout_model = keras.Sequential([
    layers.Input(shape=(28, 28)),
    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.20),
    layers.Dense(10, activation="softmax")
])

dropout_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
dropout_model.summary()


In [ ]:
dropout_history = dropout_model.fit(
    x_train, y_train,
    epochs=5,
    batch_size=128,
    validation_split=0.10,
    verbose=1
)


In [ ]:
dropout_test_loss, dropout_test_accuracy = dropout_model.evaluate(x_test, y_test, verbose=0)
print(f"Dropout test loss: {dropout_test_loss:.4f}")
print(f"Dropout test accuracy: {dropout_test_accuracy:.4%}")

results = pd.DataFrame({
    "Model": ["Baseline", "Dropout 0.20"],
    "Test Loss": [baseline_test_loss, dropout_test_loss],
    "Test Accuracy": [baseline_test_accuracy, dropout_test_accuracy]
})
display(results)
accuracy_change = dropout_test_accuracy - baseline_test_accuracy
print(f"Change in test accuracy: {accuracy_change:+.4%}")


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(baseline_history.history["val_accuracy"], label="Baseline")
plt.plot(dropout_history.history["val_accuracy"], label="Dropout 0.20")
plt.xlabel("Epoch"); plt.ylabel("Validation Accuracy")
plt.title("Experiment: Validation Accuracy Comparison")
plt.legend(); plt.grid(True); plt.show()


## 8. Conclusion

The baseline model provides a simple fully connected approach to MNIST classification. The experiment compares it with a Dropout-regularized model.

Use the measured outputs above in the report. Do not substitute assumed accuracy values because results can vary slightly with random initialization and the software/hardware environment.


In [ ]:
print("=" * 60)
print("MNIST ASSIGNMENT RESULT SUMMARY")
print("=" * 60)
print(f"Baseline test accuracy : {baseline_test_accuracy:.4%}")
print(f"Dropout test accuracy  : {dropout_test_accuracy:.4%}")
print(f"Accuracy difference    : {accuracy_change:+.4%}")
print(f"5-image accuracy       : {five_image_accuracy:.0%}")
print("=" * 60)
